# copyme prototype
1. parse personal writing as context
2. process text to structured description of linguistic style

e.g.
```python
linguistic_style_metrics = {
    "Type-Token Ratio (TTR)": {
        "description": "The ratio of unique types (distinct words) to total tokens (total words). It measures lexical diversity but is highly sensitive to text length (decreasing as text length increases).",
        "data_type": "float",
        "constraints": {"min": 0.0, "max": 1.0},
        "typical_range": "0.3 (long texts) to 0.7 (short texts)"
    },
    "Mean Length of Sentence (MLS)": {
        "description": "The average number of words per sentence (also known as Sentence Length). Higher values correlate with formal academic writing (approx. 20+ words) and syntactic complexity.",
        "data_type": "float",
        "constraints": {"min": 1.0},
        "typical_range": "15.0 - 25.0"
    },
    "Punctuation Density": {
        "description": "The frequency of punctuation marks per 1,000 words. Variations reflect syntactic pacing; low density suggests run-on sentences or spoken style, while high density implies complex clause structures.",
        "data_type": "float",
        "constraints": {"min": 0.0},
        "typical_range": "100.0 - 150.0"
    },
    "Discourse Marker Density": {
        "description": "The frequency of discourse markers (e.g., 'however', 'so', 'well') per 1,000 words. Rates above 50/1,000 are typical of casual spoken conversation; academic texts typically feature lower rates (<15/1,000).",
        "data_type": "float",
        "constraints": {"min": 0.0},
        "typical_range": "10.0 - 60.0"
    },
    "Hapax Legomena Ratio": {
        "description": "The proportion of words appearing exactly once in the text relative to the total vocabulary size. A primary indicator of descriptive richness and vocabulary breadth.",
        "data_type": "float",
        "constraints": {"min": 0.0, "max": 1.0},
        "typical_range": "0.40 - 0.60"
    },
    "Formulaic Density": {
        "description": "The proportion of the text comprised of recurrent 2-gram to 4-gram sequences (lexical bundles). High values indicate reliance on formulaic language, clichés, or 'crutch' phrases.",
        "data_type": "float",
        "constraints": {"min": 0.0, "max": 1.0},
        "typical_range": "0.10 - 0.30"
    },
    "Modal Hedging Ratio": {
        "description": "The ratio of epistemic modality markers (e.g., 'could', 'may', 'possibly') to total assertions. Measures the author's degree of certainty or tentativeness (common in scientific discussion sections).",
        "data_type": "float",
        "constraints": {"min": 0.0, "max": 1.0},
        "typical_range": "0.05 - 0.20"
    },
    "Function Word Frequency": {
        "description": "A normalized distribution vector of function words (prepositions, pronouns, determiners). These high-frequency items are content-independent and serve as reliable markers for authorship attribution (stylometry).",
        "data_type": "dict[str, float]",
        "constraints": {"value_min": 0.0, "value_max": 1.0}
    },
    "Flesch Reading Ease": {
        "description": "A standardized readability score based on average sentence length and average syllables per word. Scores 90-100 are 'Very Easy'; scores 0-30 are 'Very Difficult' (academic/professional).",
        "data_type": "float",
        "constraints": {"min": -20.0, "max": 120.0}, 
        "typical_range": "30.0 - 70.0"
    },
    "Lexical Sophistication": {
        "description": "Categorical assessment of vocabulary rarity based on frequency lists (e.g., academic word lists).",
        "data_type": "str",
        "allowed_values": ["basic", "intermediate", "advanced", "specialized/technical"]
    },
    "Syntactic Variety": {
        "description": "Assessment of the variation in sentence structures (simple, compound, complex, compound-complex) and clause types.",
        "data_type": "str",
        "allowed_values": ["repetitive", "standard", "varied", "complex"]
    },
    "Cohesive Harmony": {
        "description": "A measure of the interaction between cohesive chains (identity chains and similarity chains) that facilitate text continuity, as defined by Ruqaiya Hasan.",
        "data_type": "str",
        "allowed_values": ["fragmented", "loose", "coherent", "highly integrated"]
    },
    "Rhetorical Intent": {
        "description": "The primary communicative purpose of the text, influencing the choice of stance and register.",
        "data_type": "str",
        "allowed_values": ["informative", "persuasive", "expository", "narrative", "descriptive", "instructional"]
    },
    "Genre Alignment": {
        "description": "The extent to which the text's stylistic features match the prototypical conventions of its intended genre.",
        "data_type": "str",
        "allowed_values": ["academic", "legal", "literary_fiction", "journalistic", "technical_manual", "conversational_speech"]
    }
}
```

In [1]:
import spacy
import textstat
from empath import Empath
from collections import Counter

## Prototype example of generating linquistic metrics

In [ ]:
import spacy
import textstat
from empath import Empath
from collections import Counter

# --- INITIALIZATION ---
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Error: Model not found. Run: python -m spacy download en_core_web_sm")
    exit()

lexicon = Empath()

def analyze_linguistic_style(text):
    """
    Analyzes text to return a dictionary of quantitative metrics 
    and qualitative heuristics (using Empath for tone/intent).
    """
    doc = nlp(text)
    
    # --- 1. BASIC PRE-PROCESSING ---
    words = [token.text for token in doc if not token.is_punct and not token.is_space]
    word_count = len(words)
    sentences = list(doc.sents)
    sentence_count = len(sentences)
    
    if word_count == 0: return None

    # --- 2. QUANTITATIVE METRICS ---
    
    # Type-Token Ratio (TTR)
    unique_words = set(word.lower() for word in words)
    ttr = len(unique_words) / word_count

    # Mean Length of Sentence (MLS)
    mls = word_count / sentence_count

    # Punctuation Density (per 1000 words)
    punct_count = len([token for token in doc if token.is_punct])
    punct_density = (punct_count / word_count) * 1000

    # Function Word Frequency
    pos_counts = Counter(token.pos_ for token in doc)
    total_pos = sum(pos_counts.values())
    function_word_freq = {
        "prepositions": round(pos_counts.get("ADP", 0) / total_pos, 3),
        "conjunctions": round((pos_counts.get("CCONJ", 0) + pos_counts.get("SCONJ", 0)) / total_pos, 3),
        "pronouns": round(pos_counts.get("PRON", 0) / total_pos, 3),
        "determiners": round(pos_counts.get("DET", 0) / total_pos, 3)
    }

    # Flesch Reading Ease (formerly Readability Index)
    readability = textstat.flesch_reading_ease(text)

    # --- 3. QUALITATIVE ATTRIBUTES (via EMPATH & Heuristics) ---

    # Analyze text topics with Empath
    empath_scores = lexicon.analyze(text, normalize=True) or {}

    # A. Rhetorical Intent
    if empath_scores.get('help', 0) > 0.02 or empath_scores.get('negotiate', 0) > 0.02:
        intent = "persuasive"
    elif empath_scores.get('science', 0) > 0.02 or empath_scores.get('school', 0) > 0.02:
        intent = "expository"
    elif empath_scores.get('emotional', 0) > 0.02 or empath_scores.get('negative_emotion', 0) > 0.02:
        intent = "narrative"
    elif empath_scores.get('appearance', 0) > 0.02 or empath_scores.get('art', 0) > 0.02:
        intent = "descriptive"
    elif empath_scores.get('tool', 0) > 0.02 or empath_scores.get('work', 0) > 0.02:
        intent = "instructional"
    else:
        intent = "informative"

    # B. Genre Alignment
    if readability < 40 and (empath_scores.get('law', 0) > 0.01 or empath_scores.get('government', 0) > 0.01):
        genre = "legal"
    elif readability < 50 and empath_scores.get('science', 0) > 0.02:
        genre = "academic"
    elif readability > 60 and intent == "narrative":
        genre = "literary_fiction"
    elif empath_scores.get('computer', 0) > 0.02 or empath_scores.get('technology', 0) > 0.02:
        genre = "technical_manual"
    elif intent == "persuasive" or empath_scores.get('speaking', 0) > 0.02:
        genre = "conversational_speech"
    elif 40 < readability < 60 and intent == "informative":
        genre = "journalistic"
    else:
        genre = "conversational_speech"

    # C. Lexical Sophistication
    long_words = [w for w in words if len(w) > 6]
    long_word_ratio = len(long_words) / word_count
    
    if long_word_ratio > 0.3: sophistication = "specialized/technical"
    elif long_word_ratio > 0.2: sophistication = "advanced"
    elif long_word_ratio > 0.12: sophistication = "intermediate"
    else: sophistication = "basic"

    # D. Syntactic Variety
    sent_lengths = [len([t for t in s if not t.is_punct]) for s in sentences]
    if sent_lengths:
        mean_len = sum(sent_lengths) / len(sent_lengths)
        variance = sum((x - mean_len) ** 2 for x in sent_lengths) / len(sent_lengths)
        std_dev = variance ** 0.5
    else:
        std_dev = 0
    
    if std_dev > 10: variety = "complex"
    elif std_dev > 6: variety = "varied"
    elif std_dev > 3: variety = "standard"
    else: variety = "repetitive"

    # E. Cohesive Harmony
    overlaps = 0
    if sentence_count > 1:
        for i in range(1, sentence_count):
            prev_nouns = {t.lemma_ for t in sentences[i-1] if t.pos_ in ["NOUN", "PROPN"]}
            curr_nouns = {t.lemma_ for t in sentences[i] if t.pos_ in ["NOUN", "PROPN"]}
            if not prev_nouns.isdisjoint(curr_nouns):
                overlaps += 1
        cohesion_score = overlaps / (sentence_count - 1)
    else:
        cohesion_score = 1.0 # Single sentence is perfectly cohesive with itself

    if cohesion_score > 0.7: harmony = "highly integrated"
    elif cohesion_score > 0.5: harmony = "coherent"
    elif cohesion_score > 0.25: harmony = "loose"
    else: harmony = "fragmented"


    # --- 4. COMPILE FINAL DICTIONARY ---
    return {
        "Type-Token Ratio (TTR)": round(ttr, 3),
        "Mean Length of Sentence (MLS)": round(mls, 2),
        "Punctuation Density": round(punct_density, 2),
        "Function Word Frequency": function_word_freq,
        "Flesch Reading Ease": round(readability, 1),
        "Lexical Sophistication": sophistication,
        "Syntactic Variety": variety,
        "Cohesive Harmony": harmony,
        "Rhetorical Intent": intent,
        "Genre Alignment": genre
    }

# --- EXAMPLE USAGE ---

# Sample 1: Academic / Formal
text_academic = """
The proliferation of digital technologies has fundamentally altered the landscape of modern pedagogy. 
Specifically, the integration of algorithmic assessment tools allows for granular analysis of student performance. 
However, this reliance on quantitative metrics often obscures the qualitative nuances of learning. 
Thus, educators must maintain a critical distance from these tools.
"""

# Sample 2: Narrative / Simple
text_narrative = """
I walked down the street and saw a bright light. It was strange. 
I felt scared but I kept walking anyway because I wanted to see what it was. 
Suddenly, a cat jumped out! I laughed.
"""

print("--- ACADEMIC TEXT ANALYSIS ---")
import json
print(json.dumps(analyze_linguistic_style(text_academic), indent=4))

print("\n--- NARRATIVE TEXT ANALYSIS ---")
print(json.dumps(analyze_linguistic_style(text_narrative), indent=4))


--- ACADEMIC TEXT ANALYSIS ---
{
    "Type-Token Ratio (TTR)": 0.84,
    "Mean Length of Sentence (MLS)": 12.5,
    "Punctuation Density": 140.0,
    "Function Word Frequency": {
        "prepositions": 0.129,
        "conjunctions": 0.0,
        "pronouns": 0.0,
        "determiners": 0.113
    },
    "Readability Index": 4.6,
    "Lexical Sophistication": "very high",
    "Syntactic Variety": "minimal",
    "Cohesive Harmony": "weak",
    "Rhetorical Intent": "expository",
    "Genre Alignment": "conversational"
}

--- NARRATIVE TEXT ANALYSIS ---
{
    "Type-Token Ratio (TTR)": 0.806,
    "Mean Length of Sentence (MLS)": 7.2,
    "Punctuation Density": 166.67,
    "Function Word Frequency": {
        "prepositions": 0.043,
        "conjunctions": 0.065,
        "pronouns": 0.174,
        "determiners": 0.065
    },
    "Readability Index": 96.7,
    "Lexical Sophistication": "medium",
    "Syntactic Variety": "moderate",
    "Cohesive Harmony": "weak",
    "Rhetorical Intent": "narra

In [ ]:
# Sample 3: Legal / Contractual
text_legal = """
The Party of the First Part shall, subject to the provisions herein contained, indemnify and hold harmless the Party of the Second Part against all claims, liabilities, and expenses arising from the breach of any representation or warranty made by the indemnifying party under this Agreement. This indemnification shall survive the termination of this Agreement for a period of five years.
"""

# Sample 4: Technical / Computer Science
text_technical = """
To optimize the neural network's performance, we implemented a custom gradient descent algorithm using TensorFlow and CUDA kernels. The architecture utilizes convolutional layers followed by a transformer block to capture long-range dependencies in the input data. We observed a significant reduction in latency after migrating the compute-intensive tasks to the GPU.
"""

# Sample 5: Journalistic / News
text_journalistic = """
City officials announced a new initiative today aimed at reducing traffic congestion in the downtown area. The plan includes the expansion of bike lanes and the introduction of a smart traffic management system. Residents expressed cautious optimism about the project, noting that similar efforts in the past had mixed results.
"""

print("--- LEGAL TEXT ANALYSIS ---")
import json
print(json.dumps(analyze_linguistic_style(text_legal), indent=4))

print("\n--- TECHNICAL TEXT ANALYSIS ---")
print(json.dumps(analyze_linguistic_style(text_technical), indent=4))

print("\n--- JOURNALISTIC TEXT ANALYSIS ---")
print(json.dumps(analyze_linguistic_style(text_journalistic), indent=4))

In [3]:
# --- EXTREME EXAMPLES ---

# 1. High Type-Token Ratio (TTR) - Every word is unique
text_extreme_ttr = "Linguistic diversity necessitates utilizing unique vocabulary, avoiding repetitive lexical selections, demonstrating expansive terminology."

# 2. High Mean Length of Sentence (MLS) - One extremely long, complex sentence
text_extreme_mls = "The multi-faceted nature of socioeconomic disparities within urban environments necessitates a comprehensive re-evaluation of current infrastructural paradigms, particularly concerning the equitable distribution of public resources and the mitigation of systemic barriers that historically marginalized communities encounter on a daily basis while attempting to navigate bureaucratic hurdles and access essential services that are often gatekept by opaque institutional frameworks."

# 3. High Punctuation Density - Overuse of various marks
text_extreme_punct = "Wait; what? Is this--really--necessary?! (Maybe...) Yes: absolutely, definitely, surely; however, why???"

# 4. Low Readability (Hard to read) - High syllables, complex structure
text_extreme_hard_read = "The phenomenological investigation of existential angst requires a profound ontological commitment to deciphering the primordial structures of human consciousness and its inherent teleological orientation toward transcendence within a secularized milieu."

# 5. High Syntactic Variety - Extremely different sentence lengths
text_extreme_syntax_variety = "Stop. Now, let us consider the profound implications of our actions on the future of this planet. Think. We must act before it is too late, for the consequences of our current trajectory are nothing short of catastrophic for every living being on Earth. Now!"

# 6. High Cohesive Harmony - Strong noun overlap
text_extreme_harmony = "The architect designed the building. This building features a unique glass facade. The facade reflects the surrounding city skyline. The skyline defines the character of the urban landscape."

# 7. High Persuasive Intent (Rhetorical)
text_extreme_persuasive = "We must stand together! We need your help to win this fight. Join us now, negotiate for a better future, and help us transform the world today!"

samples = {
    "EXTREME TTR": text_extreme_ttr,
    "EXTREME MLS": text_extreme_mls,
    "EXTREME PUNCTUATION": text_extreme_punct,
    "EXTREME HARD READ": text_extreme_hard_read,
    "EXTREME SYNTACTIC VARIETY": text_extreme_syntax_variety,
    "EXTREME HARMONY": text_extreme_harmony,
    "EXTREME PERSUASIVE": text_extreme_persuasive
}

for name, text in samples.items():
    print(f"--- {name} ---")
    print(json.dumps(analyze_linguistic_style(text), indent=4))
    print()

--- EXTREME TTR ---
{
    "Type-Token Ratio (TTR)": 1.0,
    "Mean Length of Sentence (MLS)": 13.0,
    "Punctuation Density": 230.77,
    "Function Word Frequency": {
        "prepositions": 0.0,
        "conjunctions": 0.0,
        "pronouns": 0.0,
        "determiners": 0.0
    },
    "Readability Index": -112.2,
    "Lexical Sophistication": "very high",
    "Syntactic Variety": "minimal",
    "Cohesive Harmony": "strong",
    "Rhetorical Intent": "informative",
    "Genre Alignment": "conversational"
}

--- EXTREME MLS ---
{
    "Type-Token Ratio (TTR)": 0.867,
    "Mean Length of Sentence (MLS)": 60.0,
    "Punctuation Density": 66.67,
    "Function Word Frequency": {
        "prepositions": 0.109,
        "conjunctions": 0.047,
        "pronouns": 0.031,
        "determiners": 0.078
    },
    "Readability Index": -72.3,
    "Lexical Sophistication": "very high",
    "Syntactic Variety": "minimal",
    "Cohesive Harmony": "strong",
    "Rhetorical Intent": "informative",
    "Ge

In [ ]:
nick_wong_cover_letter = """I recently talked to you at Chiasma Synapse on the 21/9/2016, where you and your colleague
were recruiting students who are passionate about gaining science industry experience.
I have spent this year in the lab training in molecular biology techniques related to cancer
research as part of my honours degree. I have also conducted RNA extraction and RT-qPCR
in two of my 300 level papers during my undergraduate degree. I am very resourceful, and I
constantly think of more efficient ways to complete tasks.
I initially became interested in science industry from a talk you gave in the biochemistry
department last year. Since then, I have gained insight regarding the status quo between
academia and the industry. It is evident that academia portrays a strong prestige over the
industry. This led me to approach someone that had experience in both fields, one of your
founders, Parry Guilford. Parry deeply inspired me, and as a result, I have taken up the
responsibility of being CEO for Chiasma Dunedin in 2017.
I jumped at the opportunity to apply for this internship because I could see the invaluable
experience I would gain. I believe this will be a strong kick start to my career, which will
allow me to pursue my dreams."""

print(json.dumps(analyze_linguistic_style(nick_wong_cover_letter), indent=4))



{
    "Type-Token Ratio (TTR)": 0.648,
    "Mean Length of Sentence (MLS)": 19.09,
    "Punctuation Density": 95.24,
    "Function Word Frequency": {
        "prepositions": 0.115,
        "conjunctions": 0.037,
        "pronouns": 0.131,
        "determiners": 0.066
    },
    "Readability Index": 49.1,
    "Lexical Sophistication": "high",
    "Syntactic Variety": "moderate",
    "Cohesive Harmony": "adequate",
    "Rhetorical Intent": "expository",
    "Genre Alignment": "academic"
}


In [8]:
ai_reversed_engine_text = """I am writing to express my enthusiastic interest in the Biotechnology Research Internship position at Pacific Flair, as advertised on your professional careers portal. My academic background in molecular biology, coupled with a rigorous focus on genomic sequencing, aligns with the innovative research priorities established by your organisation. Throughout my undergraduate studies, I have maintained a steadfast commitment to understanding the biochemical pathways that facilitate cellular regeneration; consequently, I am eager to apply this theoretical knowledge within a practical, high-throughput laboratory environment.
During my recent laboratory assistantship, I prioritised the meticulous optimisation of PCR protocols to ensure the integrity of diverse genetic samples. This role required a high degree of technical precision and an ability to synthesise complex data sets into coherent reports for senior researchers. The experience refined my competency in utilising CRISPR-Cas9 technologies; furthermore, it underscored the importance of methodological consistency when navigating the challenges inherent in longitudinal clinical trials. I am confident that my familiarity with these advanced analytical frameworks will allow me to contribute meaningfully to your ongoing developmental projects.
Pacific Flair’s reputation for pioneering sustainable biopharmaceuticals represents the ideal environment for an aspiring scientist dedicated to translational medicine. I am particularly drawn to your recent publications regarding enzymatic catalysis, which demonstrate a sophisticated approach to solving contemporary therapeutic hurdles. I would welcome the opportunity to discuss how my technical skills and academic dedication can support your team's objectives during a formal interview. Thank you for your time and for considering my application.
"""

print(json.dumps(analyze_linguistic_style(ai_reversed_engine_text), indent=4))

{
    "Type-Token Ratio (TTR)": 0.709,
    "Mean Length of Sentence (MLS)": 22.82,
    "Punctuation Density": 95.62,
    "Function Word Frequency": {
        "prepositions": 0.108,
        "conjunctions": 0.025,
        "pronouns": 0.094,
        "determiners": 0.076
    },
    "Readability Index": -7.1,
    "Lexical Sophistication": "very high",
    "Syntactic Variety": "moderate",
    "Cohesive Harmony": "weak",
    "Rhetorical Intent": "expository",
    "Genre Alignment": "academic"
}


In [9]:
ai_reversed_engine_text_v2 = """I am writing to apply for the Biotechnology Research Internship at Pacific Flair, as advertised on your professional careers website. My academic background in molecular biology, coupled with genomic sequencing, aligns with the innovative research goals and priorities of your organisation. Throughout my undergraduate studies, I maintained a consistent focus on the biochemical pathways that facilitate cellular regeneration within various clinical settings. Consequently, I am eager to apply this knowledge within your high-throughput laboratory environment to support your current research objectives.

In my recent assistantship, I prioritised the optimisation of PCR protocols; this ensured the long-term integrity of our genetic samples. This role required high technical precision and the ability to synthesise complex data into reports for the senior staff. Additionally, it allowed me to refine my skills in CRISPR technologies while I followed strict guidelines for various laboratory trials. Furthermore, I am certain that my familiarity with these analytical frameworks will help me contribute to your latest developmental projects.

Pacific Flair is known for pioneering sustainable biopharmaceuticals, which represents an ideal environment for an aspiring and dedicated scientist. I am particularly interested in your work regarding enzymatic catalysis, because it addresses many contemporary therapeutic hurdles through science. I look forward to discussing how my technical skills and my personal dedication can benefit your professional research team. Thank you for your time, and I appreciate your consideration of my application to join your internship programme."""

print(json.dumps(analyze_linguistic_style(ai_reversed_engine_text_v2), indent=4))

{
    "Type-Token Ratio (TTR)": 0.667,
    "Mean Length of Sentence (MLS)": 19.75,
    "Punctuation Density": 109.7,
    "Function Word Frequency": {
        "prepositions": 0.102,
        "conjunctions": 0.038,
        "pronouns": 0.136,
        "determiners": 0.049
    },
    "Readability Index": 12.0,
    "Lexical Sophistication": "very high",
    "Syntactic Variety": "minimal",
    "Cohesive Harmony": "weak",
    "Rhetorical Intent": "expository",
    "Genre Alignment": "academic"
}


In [10]:
my_custom_cover_letter_overly_excited = """
Hi team at leonardo! I have been an avid user of your ai generation tool for years now, and it is a product that I absolutely love!
I have use it extensively in my life, and it has speed up a lot of my work, especially when it comes to front end design assets. 
I see that you have an opening for a front end engineer, which is a recent career path I have decided to take. I believe that my background in
graphic design and machine learning would provide the diverse background that will enable me to excel in this role long term. 
I absolutely love learning, and continue to learn new things every day. I look to you reviewing my application. Kind regards, 
Josh"""

print(json.dumps(analyze_linguistic_style(my_custom_cover_letter_overly_excited), indent=4))


{
    "Type-Token Ratio (TTR)": 0.659,
    "Mean Length of Sentence (MLS)": 15.75,
    "Punctuation Density": 103.17,
    "Function Word Frequency": {
        "prepositions": 0.076,
        "conjunctions": 0.048,
        "pronouns": 0.159,
        "determiners": 0.062
    },
    "Readability Index": 70.7,
    "Lexical Sophistication": "medium",
    "Syntactic Variety": "diverse",
    "Cohesive Harmony": "adequate",
    "Rhetorical Intent": "informative",
    "Genre Alignment": "conversational"
}


In [15]:
teenager_vibe = """
Okay, so we had this data platform that we really wanted to get research data on, and so it was a batch job, and I really didn't know what to actually do at first, so we just had to weigh it manually to try to figure out the actual pain points that we needed to address, and we found that we don't really need any massive parallel Spark processing information. What we really need is a way to do release management and a way to manage schema deployments, schema changes, and do validation, do it all in a way that we could like trace it.
Oh my god, so I don't actually know too much about privacy and compliance, but we did try to make sure everything was secured in a VPC and that we had sufficient security checks and fine-grained access, so you couldn't have any Joe blog just come in and try to access the data and have to go through us.
"""

print(json.dumps(analyze_linguistic_style(teenager_vibe), indent=4))


{
    "Type-Token Ratio (TTR)": 0.538,
    "Mean Length of Sentence (MLS)": 56.33,
    "Punctuation Density": 88.76,
    "Function Word Frequency": {
        "prepositions": 0.043,
        "conjunctions": 0.075,
        "pronouns": 0.134,
        "determiners": 0.053
    },
    "Readability Index": 34.2,
    "Lexical Sophistication": "medium",
    "Syntactic Variety": "highly diverse",
    "Cohesive Harmony": "weak",
    "Rhetorical Intent": "informative",
    "Genre Alignment": "technical"
}


In [12]:
import nltk

# Trying version 2, with more personal stuff

In [3]:
import spacy
import nltk
import json
import textstat
from empath import Empath
from collections import Counter

# --- INITIALIZATION ---

# 1. Load Spacy Model
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Error: Spacy model not found. Run: python -m spacy download en_core_web_sm")
    exit()

# 2. Load NLTK Resources
try:
    from nltk.corpus import nps_chat
    try:
        chat_words = nps_chat.tagged_words()
        NLTK_FILLERS = {word.lower() for word, tag in chat_words if tag == 'UH'}
    except LookupError:
        print("Warning: NLTK 'nps_chat' not found. Using fallback list.")
        NLTK_FILLERS = set()
except ImportError:
    NLTK_FILLERS = set()

FALLBACK_MARKERS = {
    "so", "actually", "really", "okay", "well", "like", "basically", 
    "literally", "totally", "honestly", "anyway", "you know", "i mean", "right",
    "however", "moreover", "therefore", "thus", "hence", "furthermore"
}
DISCOURSE_MARKERS = NLTK_FILLERS.union(FALLBACK_MARKERS)

lexicon = Empath()


# --- UNIFIED ANALYZER CLASS ---

class StyleAnalyzer:
    def __init__(self, text):
        self.text = text
        self.doc = nlp(text)
        self.words = [token.text for token in self.doc if not token.is_punct and not token.is_space]
        self.sentences = list(self.doc.sents)
        self.word_count = len(self.words)
        self.sentence_count = len(self.sentences)
        self.unique_words = set(w.lower() for w in self.words)
        self.empath_scores = lexicon.analyze(text, normalize=True) or {}
        self.repeated_counts = self._get_repeated_ngram_counts()

    def _get_ngrams(self, n):
        return [" ".join(self.words[i:i+n]).lower() for i in range(len(self.words)-n+1)]

    def _get_repeated_ngram_counts(self, n_range=[2, 3, 4]):
        combined_counts = Counter()
        for n in n_range:
            ngrams = self._get_ngrams(n)
            counts = Counter(ngrams)
            for gram, count in counts.items():
                if count > 1:
                    combined_counts[gram] += count
        return combined_counts

    # --- QUANTITATIVE METRICS ---

    def get_ttr(self):
        if self.word_count == 0: return 0.0
        return round(len(self.unique_words) / self.word_count, 3)

    def assess_ttr(self, ttr):
        if ttr > 0.6: return "diverse"
        elif ttr > 0.4: return "standard"
        return "repetitive"

    def get_mls(self):
        if self.sentence_count == 0: return 0.0
        return round(self.word_count / self.sentence_count, 2)

    def assess_mls(self, mls):
        if mls > 25: return "complex/elaborate"
        elif mls > 15: return "standard"
        return "short/telegraphic"

    def get_punctuation_density(self):
        if self.word_count == 0: return 0.0
        punct_count = len([t for t in self.doc if t.is_punct])
        return round((punct_count / self.word_count) * 1000, 2)

    def assess_punctuation_density(self, density):
        if density > 150: return "heavily punctuated"
        elif density > 100: return "balanced"
        return "sparse"

    def get_flesch_reading_ease(self):
        return round(textstat.flesch_reading_ease(self.text), 1)

    def assess_flesch_reading_ease(self, score):
        if score > 80: return "very easy"
        elif score > 60: return "easy/standard"
        elif score > 40: return "difficult"
        return "very difficult/technical"

    def get_discourse_marker_density(self):
        if self.word_count == 0: return 0.0
        dm_count = sum(1 for w in self.words if w.lower() in DISCOURSE_MARKERS)
        return round((dm_count / self.word_count) * 1000, 2)

    def assess_discourse_marker_density(self, density):
        if density > 40: return "conversational"
        elif density > 15: return "natural"
        return "formal/polished"

    def get_hapax_legomena_ratio(self):
        total_unique = len(self.unique_words)
        if total_unique == 0: return 0.0
        word_freqs = Counter(w.lower() for w in self.words)
        hapax_count = sum(1 for count in word_freqs.values() if count == 1)
        return round(hapax_count / total_unique, 3)

    def assess_hapax_legomena_ratio(self, ratio):
        if ratio > 0.6: return "unique/creative"
        elif ratio > 0.4: return "rich"
        return "basic/repetitive"

    def get_formulaic_density(self):
        if self.word_count == 0: return 0.0
        total_repeated = sum(self.repeated_counts.values())
        return round(min(total_repeated / self.word_count, 1.0), 3)

    def assess_formulaic_density(self, density):
        if density > 0.3: return "highly formulaic"
        elif density > 0.1: return "standard"
        return "original"

    def get_modal_ratio(self):
        hedges = {
            "can", "could", "may", "might", "would", "should", 
            "possibly", "probably", "likely", "maybe", "seems", "appear", "suggests"
        }
        hedge_count = sum(1 for t in self.doc if t.text.lower() in hedges or t.lemma_.lower() in hedges)
        if self.sentence_count == 0: return 0.0
        return round(min(hedge_count / self.sentence_count, 1.0), 2)

    def assess_modal_ratio(self, ratio):
        if ratio > 0.2: return "tentative/hedged"
        elif ratio > 0.05: return "balanced"
        return "assertive/direct"

    def get_function_word_frequencies(self):
        pos_counts = Counter(token.pos_ for token in self.doc)
        total = sum(pos_counts.values()) or 1
        
        def get_pos_ratio(target_pos_list):
            count = sum(pos_counts.get(pos, 0) for pos in target_pos_list)
            return round(count / total, 3)

        return {
            "prepositions": get_pos_ratio(["ADP"]),
            "conjunctions": get_pos_ratio(["CCONJ", "SCONJ"]),
            "pronouns": get_pos_ratio(["PRON"]),
            "determiners": get_pos_ratio(["DET"])
        }

    # --- WORDS DATA ---

    def get_common_vocabulary(self, n=10):
        word_freqs = Counter(w.lower() for w in self.words)
        return [word for word, count in word_freqs.most_common(n)]

    def get_frequent_phrases(self, n=10):
        return [phrase for phrase, count in self.repeated_counts.most_common(n)]

    # --- QUALITATIVE ANALYZER ---

    def get_lexical_sophistication(self):
        if not self.words: return "basic"
        long_words = [w for w in self.words if len(w) > 6]
        ratio = len(long_words) / self.word_count
        
        if ratio > 0.3: return "specialized/technical"
        elif ratio > 0.2: return "advanced"
        elif ratio > 0.12: return "intermediate"
        return "basic"

    def get_syntactic_variety(self):
        if not self.sentences: return "repetitive"
        sent_lengths = [len([t for t in s if not t.is_punct]) for s in self.sentences]
        if not sent_lengths: return "repetitive"
        
        mean_len = sum(sent_lengths) / len(sent_lengths)
        variance = sum((x - mean_len) ** 2 for x in sent_lengths) / len(sent_lengths)
        std_dev = variance ** 0.5
        
        if std_dev > 10: return "complex"
        elif std_dev > 6: return "varied"
        elif std_dev > 3: return "standard"
        return "repetitive"

    def get_cohesive_harmony(self):
        if self.sentence_count <= 1: return "coherent"
        overlaps = 0
        for i in range(1, self.sentence_count):
            prev_nouns = {t.lemma_ for t in self.sentences[i-1] if t.pos_ in ["NOUN", "PROPN"]}
            curr_nouns = {t.lemma_ for t in self.sentences[i] if t.pos_ in ["NOUN", "PROPN"]}
            if not prev_nouns.isdisjoint(curr_nouns):
                overlaps += 1
                
        score = overlaps / (self.sentence_count - 1)
        if score > 0.7: return "highly_integrated"
        elif score > 0.5: return "coherent"
        elif score > 0.25: return "loose"
        return "fragmented"

    def get_rhetorical_intent(self):
        scores = self.empath_scores
        if scores.get('help', 0) > 0.02 or scores.get('negotiate', 0) > 0.02:
            return "persuasive"
        elif scores.get('science', 0) > 0.02 or scores.get('school', 0) > 0.02:
            return "expository"
        elif scores.get('emotional', 0) > 0.02 or scores.get('negative_emotion', 0) > 0.02:
            return "narrative"
        elif scores.get('appearance', 0) > 0.02 or scores.get('art', 0) > 0.02:
            return "descriptive"
        elif scores.get('tool', 0) > 0.02 or scores.get('work', 0) > 0.02:
            return "instructional"
        return "informative"

    def get_genre_alignment(self, intent):
        readability = textstat.flesch_reading_ease(self.text)
        scores = self.empath_scores
        
        if readability < 40 and (scores.get('law', 0) > 0.01 or scores.get('government', 0) > 0.01):
            return "legal"
        elif readability < 50 and scores.get('science', 0) > 0.02:
            return "academic"
        elif readability > 60 and intent == "narrative":
            return "literary_fiction"
        elif scores.get('computer', 0) > 0.02 or scores.get('technology', 0) > 0.02:
            return "technical_manual"
        elif intent == "persuasive" or scores.get('speaking', 0) > 0.02:
            return "conversational_speech"
        elif 40 < readability < 60 and intent == "informative":
            return "journalistic"
        return "conversational_speech"


# --- MAIN ANALYSIS FUNCTION ---

def analyze_linguistic_style(text):
    analyzer = StyleAnalyzer(text)
    
    # Calculate Quantitative Values
    ttr = analyzer.get_ttr()
    mls = analyzer.get_mls()
    punct_density = analyzer.get_punctuation_density()
    dm_density = analyzer.get_discourse_marker_density()
    hapax = analyzer.get_hapax_legomena_ratio()
    formulaic = analyzer.get_formulaic_density()
    modal = analyzer.get_modal_ratio()
    readability = analyzer.get_flesch_reading_ease()
    
    # Calculate Intent for alignment
    intent = analyzer.get_rhetorical_intent()
    
    return {
        "quantitative": {
            "type_token_ratio": ttr,
            "mean_length_of_sentence": mls,
            "punctuation_density": punct_density,
            "discourse_marker_density": dm_density,
            "hapax_legomena_ratio": hapax,
            "formulaic_density": formulaic,
            "modal_hedging_ratio": modal,
            "flesch_reading_ease": readability,
            "function_word_frequency": analyzer.get_function_word_frequencies()
        },
        "qualitative": {
            "type_token_ratio_assessment": analyzer.assess_ttr(ttr),
            "mean_length_of_sentence_assessment": analyzer.assess_mls(mls),
            "punctuation_density_assessment": analyzer.assess_punctuation_density(punct_density),
            "discourse_marker_density_assessment": analyzer.assess_discourse_marker_density(dm_density),
            "hapax_legomena_ratio_assessment": analyzer.assess_hapax_legomena_ratio(hapax),
            "formulaic_density_assessment": analyzer.assess_formulaic_density(formulaic),
            "modal_hedging_ratio_assessment": analyzer.assess_modal_ratio(modal),
            "flesch_reading_ease_assessment": analyzer.assess_flesch_reading_ease(readability),
            "lexical_sophistication": analyzer.get_lexical_sophistication(),
            "syntactic_variety": analyzer.get_syntactic_variety(),
            "cohesive_harmony": analyzer.get_cohesive_harmony(),
            "rhetorical_intent": intent,
            "genre_alignment": analyzer.get_genre_alignment(intent)
        },
        "words": {
            "common_vocabulary": analyzer.get_common_vocabulary(),
            "frequent_phrases": analyzer.get_frequent_phrases()
        }
    }

In [4]:
teenager_vibe = """
Okay, so we had this data platform that we really wanted to get research data on, and so it was a batch job, and I really didn't know what to actually do at first, so we just had to weigh it manually to try to figure out the actual pain points that we needed to address, and we found that we don't really need any massive parallel Spark processing information. What we really need is a way to do release management and a way to manage schema deployments, schema changes, and do validation, do it all in a way that we could like trace it.
Oh my god, so I don't actually know too much about privacy and compliance, but we did try to make sure everything was secured in a VPC and that we had sufficient security checks and fine-grained access, so you couldn't have any Joe blog just come in and try to access the data and have to go through us.
"""

print(json.dumps(analyze_linguistic_style(teenager_vibe), indent=4))


{
    "quantitative": {
        "type_token_ratio": 0.538,
        "mean_length_of_sentence": 56.33,
        "punctuation_density": 88.76,
        "discourse_marker_density": 153.85,
        "hapax_legomena_ratio": 0.681,
        "formulaic_density": 0.16,
        "modal_hedging_ratio": 0.67,
        "flesch_reading_ease": 34.2,
        "function_word_frequency": {
            "prepositions": 0.043,
            "conjunctions": 0.075,
            "pronouns": 0.134,
            "determiners": 0.053
        }
    },
    "qualitative": {
        "type_token_ratio_assessment": "standard",
        "mean_length_of_sentence_assessment": "complex/elaborate",
        "punctuation_density_assessment": "sparse",
        "discourse_marker_density_assessment": "conversational",
        "hapax_legomena_ratio_assessment": "unique/creative",
        "formulaic_density_assessment": "standard",
        "modal_hedging_ratio_assessment": "tentative/hedged",
        "flesch_reading_ease_assessment": "very d

In [5]:
unconfident_fisherman = """
Dear Mr Captain, or whoever it is yourself that’s reading this,

Sorry to be bothering you with a letter, but I’m only after seeing your notice for a deckhand, and I thought I might chance my arm sending this in. To tell you the truth, I’ve been staring at the paper for an hour wondering if I should write at all, as I know a big boat like yours is serious business, and I wouldn’t want you thinking I’m trying to be something I’m not. I’m from just outside Dingle, a small spot really, and while I’ve spent a fair bit of time on my uncle’s punt, that’s only a small wooden thing for potting near the shore, not a proper vessel like the one you have, which looks fierce impressive, so it does.

I’m young enough, I’m only after finishing my school days, and this would be my first job, my first real one anyway. I’m a quiet lad, I don’t have much to say for myself usually, but I’m grand with the wet and the cold, sure being from Kerry I’m fairly used to being soaked to the bone. I might be a bit shaky on my sea legs starting off, and I’m sorry in advance if I ask a foolish question or two, but I swear on my life I’ll listen to whatever you tell me. I’m not one for messing about, and I’ve a good pair of hands on me, or so my Mam tells me anyway.

I’d be happy enough just scrubbing the deck or making the tea until I learn the ropes, I wouldn’t be looking to get in anyone’s way. I just want to work, and I’d be terrible grateful if you’d give a young fella a go. Thanks a million for your time, sorry again for the long note.

Yours,

Seán O’Connor"""

print(json.dumps(analyze_text_style(unconfident_fisherman), indent=4))


NameError: name 'analyze_text_style' is not defined

In [6]:
my_text = """
I initially joined my current role as a research data manager, as my industry did not know what a data engineer does. But I could see that this cardiovascular disease research platform would definately require a data engineer to manage the diversity of clinical data and complex unstructured scientific data. A key thing I noticed about the project was the importance for reprodubility, data provanance, and the also that it was really difficult to create a data model that could harmonise clinical data from highly dispirate studies. I realised early on, that clinical data needed to be cleaned transformed, validated to the data model, then loaded into our platform. But our process was iterative, and as each new dataset came on board, we had to accomodate for new types of data they had. It was impossible for us to predict all the possible data types before hand, so I know early on that I needed to build a system that could rapidly release a new batch of transformed data every time we did a data model / schema change. Also, I realied that reproduibility was very important for researchers so each batch of data must be linked to the schema it was transformed to, and its storage and location must be cataloged. To address this, I build a data lakehouse architecture in AWS using a combination of Glue, Athena, DBT, and Icerberg tables, all of this was orchestrated with step functions, which allowed us to push a data release into production every day if we wanted to, before it would take minimum a week. This system has been pivitol for ensuring the quality, reproducibility, and maintainability of the system. It is something that serves me very well and frees up a lot of time due to productivity gains, and it is something I am really proud of."""

print(json.dumps(analyze_linguistic_style(my_text), indent=4))

{
    "quantitative": {
        "type_token_ratio": 0.516,
        "mean_length_of_sentence": 30.6,
        "punctuation_density": 104.58,
        "discourse_marker_density": 104.58,
        "hapax_legomena_ratio": 0.665,
        "formulaic_density": 0.16,
        "modal_hedging_ratio": 0.5,
        "flesch_reading_ease": 40.0,
        "function_word_frequency": {
            "prepositions": 0.086,
            "conjunctions": 0.062,
            "pronouns": 0.109,
            "determiners": 0.086
        }
    },
    "qualitative": {
        "type_token_ratio_assessment": "standard",
        "mean_length_of_sentence_assessment": "complex/elaborate",
        "punctuation_density_assessment": "balanced",
        "discourse_marker_density_assessment": "conversational",
        "hapax_legomena_ratio_assessment": "unique/creative",
        "formulaic_density_assessment": "standard",
        "modal_hedging_ratio_assessment": "tentative/hedged",
        "flesch_reading_ease_assessment": "very 